# K_𝖖-algebras — a researcher's quick start

**What this is.** A small, dependency-free Python library that computes the
**K-theoretic Coulomb branch algebra** `A_𝖖[T]` of a 4d 𝒩=2 theory `T` — the
algebra of its half-BPS line defects, following Ambrosino–Gaiotto
(DESY-25-035). Give it a theory and it produces the algebra explicitly: the
**generators and relations**, the **structure constants**, the **Schur
indices** `I_{a,b}(𝖖)` of line defects, and the `ρ`-automorphism and trace.

**Who it's for.** Anyone who wants to *compute with* these algebras without
reading the source. This page runs the **real library in your browser** (via
WebAssembly) — nothing to install; every cell is copy-pasteable Python.

Below: one theory (the **pentagon**) worked out in full, then how to do the
same for **your own theory**.

In [1]:
# --- One-time setup: put the library on the import path --------------------
import sys, pathlib
_here = pathlib.Path.cwd()
for _base in (_here, *_here.parents):
    if (_base / "src" / "core").is_dir():
        sys.path[:0] = [str(p) for p in (_base / "src").rglob("*")
                        if p.is_dir() and p.name != "__pycache__"]
        break
print("library ready")

library ready

## A worked example — the pentagon (`A₂` Argyres–Douglas)

The simplest non-Lagrangian theory. Its algebra is generated by **five line
defects** `L₀, …, L₄` (indices mod 5), subject to the **pentagon relations**

$$L_i \, L_{i+2} \;=\; 1 + 𝖖^{-1} L_{i+1}, \qquad L_{i+1}\, L_i \;=\; 𝖖^{2}\, L_i \, L_{i+1}.$$

This is the quantum `A₂` cluster algebra. The library builds it and multiplies
in it — here is the defining relation as a computed product, and the full
multiplication table of the generators.

In [2]:
from samples import PentagonSampleKAlgebra
P = PentagonSampleKAlgebra()

L   = [(i, 1, 0) for i in range(5)]     # the five line defects L_0 .. L_4
one = (0, 0, 0)                         # the unit

# Readable names for the low-lying canonical basis, for printing:
names = {one: "1"}
for i in range(5):
    names[(i, 1, 0)] = f"L{i}"
    names[(i, 2, 0)] = f"L{i}^2"
    names[(i, 1, 1)] = f"L{i}L{(i + 1) % 5}"
def show(elem):
    out = []
    for lab, poly in elem.terms.items():
        tag = names.get(tuple(lab), f"L_{tuple(lab)}")
        p = str(poly)
        out.append(tag if p == "1" else f"{p}·{tag}")
    return " + ".join(out) if out else "0"

# The defining pentagon relation, as a computed product:
print("pentagon relation:  L0·L2 =", show(P.multiply(L[0], L[2])))    # 1 + q^-1·L1

# The full multiplication table  L_i · L_j:
print("\nmultiplication table  L_i · L_j:")
for i in range(5):
    print(f"  L{i}: " + "  ;  ".join(show(P.multiply(L[i], L[j])) for j in range(5)))

pentagon relation:  L0·L2 = q^-1·L1 + 1

multiplication table  L_i · L_j:
  L0: L0^2  ;  q^-1·L0L1  ;  q^-1·L1 + 1  ;  q·L4 + 1  ;  q·L4L0
  L1: q·L0L1  ;  L1^2  ;  q^-1·L1L2  ;  q^-1·L2 + 1  ;  q·L0 + 1
  L2: q·L1 + 1  ;  q·L1L2  ;  L2^2  ;  q^-1·L2L3  ;  q^-1·L3 + 1
  L3: q^-1·L4 + 1  ;  q·L2 + 1  ;  q·L2L3  ;  L3^2  ;  q^-1·L3L4
  L4: q^-1·L4L0  ;  q^-1·L0 + 1  ;  q·L3 + 1  ;  q·L3L4  ;  L4^2

### The Schur indices, and the orthonormality conjecture

`I_{i,j}(𝖖) = Tr(ρ(L_i)·L_j)` is the **Schur index** of the pair of line defects
`(L_i, L_j)`. Collected into a matrix, its **leading (`𝖖⁰`) term is the
identity** — this is the central *orthonormality* conjecture,
`I_{i,j} = δ_{i,j} + O(𝖖)`. The theory's own **vacuum Schur index** is the
character of the `(2,5)` minimal model (Lee–Yang).

In [3]:
# The table of Schur indices I_ij = Tr(rho(L_i)·L_j); its leading term:
print("Schur-index table  I_ij[q^0]  (leading coefficient):")
print("        " + "   ".join(f"L{j}" for j in range(5)))
for i in range(5):
    print(f"  L{i}     " + "    ".join(str(P.inner_product(L[i], L[j], 6)[0])
                                       for j in range(5)))

# ...the full series are delta_ij + O(q):
print("\n  I_00 =", P.inner_product(L[0], L[0], 8), "  (diagonal:     1 + O(q))")
print("  I_02 =", P.inner_product(L[0], L[2], 8), "  (off-diagonal: 0 + O(q))")

# The theory's own vacuum Schur index — the (2,5)/Lee-Yang character:
print("\nvacuum Schur index :", P.trace(one, K=16))

Schur-index table  I_ij[q^0]  (leading coefficient):
        L0   L1   L2   L3   L4
  L0     1    0    0    0    0
  L1     0    1    0    0    0
  L2     0    0    1    0    0
  L3     0    0    0    1    0
  L4     0    0    0    0    1

  I_00 = 1 - q^2 + q^4 + q^6 + O(q^9)   (diagonal:     1 + O(q))
  I_02 = q^2 + O(q^9)   (off-diagonal: 0 + O(q))

vacuum Schur index : 1 + q^4 + q^6 + q^8 + q^10 + 2*q^12 + 2*q^14 + 3*q^16 + O(q^17)

## Do the same for *your own* theory

Any 4d 𝒩=2 theory with a **BPS quiver** works: give the antisymmetric **Dirac
pairing** `B` and the **node charges**, and the library discovers the
canonical basis and computes with it exactly as above. **Flavour symmetries
are detected automatically** (from `ker B`) and their fugacities `μ` appear in
the trace.

In [4]:
from bps_kalgebra import BPSKAlgebra

# The pentagon again, now straight from its A2 BPS quiver (same algebra):
T = BPSKAlgebra(pairing=[[0, 1], [-1, 0]], node_charges=[(1, 0), (0, 1)])
print("structure constant :", T.multiply((1, 0), (0, 1)))     # (q)·L_(1,1)
print("axioms to q^6      :", T.verify_canonical_basis(K=6))  # incl. orthonormality

# A flavoured theory: the 'hexagon' has ker(B) = Z*(1,1,1), one U(1) flavour;
# its characters mu^{±k} (shown [(±k,)]) appear in the trace:
H = BPSKAlgebra(pairing=[[0, 1, -1], [-1, 0, 1], [1, -1, 0]],
                node_charges=[(1, 0, 0), (0, 1, 0), (0, 0, 1)])
print("flavour ring       :", H.coefficient_ring())           # R(U(1)) = Z[mu^±1]
print("flavoured trace    :", H.trace((0, 0, 0), K=4))

structure constant : (q)*L_(1, 1)
axioms to q^6      : {'unital': True, 'multiplicative': True, 'bar_invariant': True, 'orthonormality': True}
flavour ring       : AbelianZPlusRing(rank=1)
flavoured trace    : 1 + ([(-1,)] + 1 + [(1,)])*q^2 + (2*[(-1,)] + [(-2,)] + 3 + 2*[(1,)] + [(2,)])*q^4 + O(q^5)

Many **named families** have direct constructors too — a rough map from physics
to code:

| theory | how to build it |
|---|---|
| `A₂` Argyres–Douglas (pentagon) | `BPSKAlgebra([[0,1],[-1,0]], [(1,0),(0,1)])` |
| SQED with `N_f` hypers (`U(1)+N_f`) | `SQEDNfSampleKAlgebra(N_f)` — flavour `SU(N_f)` |
| pure `U(N)` gauge theory | `PureUNKAlgebra(N, default_rays(N))` |
| skein algebra of a surface (class-S A₁) | `SkeinKAlgebra.polygon(n)` |
| **any** BPS-quiver theory | `BPSKAlgebra(pairing, node_charges)` |

In [5]:
# SQED with N_f flavours (a Lagrangian gauge theory), flavour symmetry SU(N_f):
from samples import SQEDNfSampleKAlgebra
print("SQED_2 flavour ring :", SQEDNfSampleKAlgebra(2).coefficient_ring())   # R(SU(2))

# Pure U(2) gauge theory: Wilson lines fuse as reps (fund (x) fund = Sym^2 (+) det),
# and its canonical basis is orthonormal:
from pure_un_kalgebra import PureUNKAlgebra, default_rays
U2 = PureUNKAlgebra(2, default_rays(2), max_len=2, K=8)
W = ((0, 0), (1, 0))                                          # the fundamental Wilson line
print("U(2) Wilson fusion  :", U2.multiply(W, W))             # L_(2,0) + L_(1,1)
print("U(2) orthonormal    :", U2.verify_orthonormality(((1, 0), (0, 0)),
                                                         ((1, 0), (0, 0)), 4))

# The pentagon realised skein-side (from curves on a surface) — same Schur index:
from skein_kalgebra import PentagonSkeinKAlgebra
print("skein Schur index   :", PentagonSkeinKAlgebra().inner_product((1, 0), (1, 0), K=6))

SQED_2 flavour ring : SUNZPlusRing(2)
U(2) Wilson fusion  : L_((0, 0), (1, 1)) + L_((0, 0), (2, 0))
U(2) orthonormal    : True
skein Schur index   : 1 - q^2 + q^4 + q^6 + O(q^7)

## Where to go next

- **Change the numbers above** — edit any Dirac pairing / node charges / labels
  and press **Run**. It is the real library, so you are computing real
  structure constants and Schur indices.
- **Full package + docs.** Per-topic guides live in `docs/` (the contract, the
  cone / RG / BPS / abelianized / skein realisations, and how the axioms fix
  the trace). Pure Python 3 — nothing to install.
- **No-code tool.** A companion web applet lets you draw a BPS quiver and read
  off its data without writing Python.

All arithmetic is **exact** in `Z[𝖖^±]`; the `O(𝖖)` is applied only when a
result is read out. The two conjectures the library is built to test — and to
*construct* algebras from — are orthonormality `I_{i,j} = δ_{i,j} + O(𝖖)`
(seen in the index table above) and RG-flow intertwining `F·S = X_γ + O(𝖖)`.